# 📄 Job Summariser — Fixed Google Colab Notebook

This notebook builds a **Deep Learning Job Description Summariser** using `t5-small`.

It works with:
- `indeed-jobs-data.xlsx` using the `description` column
- `data_jobs.xlsx` by creating a job text from available columns

Output: a short summary of a job description in bullet points.

> Run cells from top to bottom in Google Colab. Upload your Excel file when asked.

## 1) Install libraries

In [ ]:
!pip install -q transformers datasets evaluate rouge_score sentencepiece openpyxl accelerate

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.7 MB/s eta 0:00:00


## 2) Imports

In [ ]:
import os
import re
import json
import random
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

## 3) Upload and load dataset

In [ ]:
DATA_PATH = "data_jobs.xlsx"   # ← keep data_jobs.xlsx in the same folder

df = pd.read_excel(DATA_PATH)

print("Columns:", df.columns.tolist())

df["job_description"] = (
    "Job title: " + df["job_title"].astype(str) + "\n" +
    "Short title: " + df["job_title_short"].astype(str) + "\n" +
    "Skills: " + df["job_skills"].astype(str) + "\n" +
    "Type skills: " + df["job_type_skills"].astype(str)
)

df = df[["job_description"]]
df["job_description"] = df["job_description"].astype(str).str.strip()

df = df[df["job_description"] != ""]

print(f"✅ {len(df)} job descriptions loaded")
df.head()

## 4) Prepare job text

In [ ]:
DATA_PATH = "data_jobs.xlsx"

df = pd.read_excel(DATA_PATH)

print("Columns:", df.columns.tolist())
print("Rows before cleaning:", len(df))


def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def build_job_text(row):
    pieces = []

    mapping = {
        "job_title": "Job title",
        "job_title_short": "Category",
        "company_name": "Company",
        "job_location": "Location",
        "job_schedule_type": "Schedule",
        "job_work_from_home": "Work from home",
        "job_country": "Country",
        "job_skills": "Skills",
        "job_type_skills": "Skill groups",
        "salary_year_avg": "Average yearly salary",
        "salary_hour_avg": "Average hourly salary",
    }

    for col, label in mapping.items():
        if col in row.index:
            value = row[col]
            if pd.notna(value) and str(value).strip() not in ["", "nan", "None"]:
                pieces.append(f"{label}: {value}")

    return clean_text(". ".join(pieces))


df["job_text"] = df.apply(build_job_text, axis=1)

df = df[df["job_text"].str.len() >= 20].copy()
df = df.drop_duplicates(subset=["job_text"]).reset_index(drop=True)

print("✅ Usable rows:", len(df))

df[["job_text"]].head()

## 5) Create pseudo summaries

Because the dataset does not contain human-written summaries, this notebook creates **pseudo labels** using rule-based extraction. Then T5 learns to generate similar concise summaries.

In [ ]:
KEYWORDS = [
    'experience', 'required', 'requirements', 'responsibilities', 'responsible',
    'skills', 'qualification', 'degree', 'years', 'knowledge', 'proficient',
    'python', 'sql', 'java', 'cloud', 'aws', 'azure', 'data', 'machine learning',
    'communication', 'manage', 'develop', 'design', 'build', 'support', 'analysis'
]


def split_sentences(text):
    text = clean_text(text)
    sentences = re.split(r'(?<=[.!?])\s+', text)
    return [s.strip() for s in sentences if len(s.strip()) > 25]


def make_pseudo_summary(text, max_bullets=5):
    sentences = split_sentences(text)
    if not sentences:
        return "- Summary unavailable."

    scored = []
    for i, s in enumerate(sentences):
        lower = s.lower()
        score = sum(1 for kw in KEYWORDS if kw in lower)
        score += max(0, 3 - i) * 0.2  # small bonus for early sentences
        scored.append((score, i, s))

    selected = sorted(scored, key=lambda x: (-x[0], x[1]))[:max_bullets]
    selected = sorted(selected, key=lambda x: x[1])

    bullets = []
    for _, _, s in selected:
        s = s[:230].strip()
        bullets.append(f"- {s}")

    return "\n".join(bullets)


df['summary'] = df['job_text'].apply(make_pseudo_summary)

print(df[['job_text', 'summary']].iloc[0]['summary'])

## 6) Limit dataset size for Colab speed

In [ ]:
# For quick testing, use 500 rows. Increase this if you have GPU time.
MAX_ROWS = 500

if len(df) > MAX_ROWS:
    df_model = df.sample(MAX_ROWS, random_state=SEED).reset_index(drop=True)
else:
    df_model = df.copy()

print('Rows used for training:', len(df_model))

## 7) Train / validation split

In [ ]:
train_df, val_df = train_test_split(df_model[['job_text', 'summary']], test_size=0.15, random_state=SEED)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print('Train:', train_df.shape)
print('Validation:', val_df.shape)

## 8) Load T5 model

In [ ]:
MODEL_NAME = 't5-small'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

print('Loaded:', MODEL_NAME)

## 9) Tokenization

In [ ]:
MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 160
PREFIX = "summarize job description: "


def preprocess_function(batch):
    inputs = [PREFIX + str(text) for text in batch['job_text']]
    targets = [str(summary) for summary in batch['summary']]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding=False,
    )

    labels = tokenizer(
        text_target=targets,
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding=False,
    )

    model_inputs['labels'] = labels['input_ids']
    return model_inputs

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

tokenized_train = train_dataset.map(preprocess_function, batched=True, remove_columns=train_dataset.column_names)
tokenized_val = val_dataset.map(preprocess_function, batched=True, remove_columns=val_dataset.column_names)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

print(tokenized_train[0].keys())

## 10) Train the model

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir='./job_summarizer_t5',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=3e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=2,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    logging_steps=25,
    report_to='none',
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    data_collator=data_collator,
)

trainer.train()

## 11) Save model

In [ ]:
SAVE_DIR = './job_summarizer_t5_final'
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print('Model saved to:', SAVE_DIR)

## 12) Summarize a job description

In [ ]:
import re

def extract_field(text, field_name):
    pattern = field_name + r":\s*(.*?)(?:\.\s[A-Z][A-Za-z ]+:|$)"
    match = re.search(pattern, text)
    return match.group(1).strip() if match else ""


def summarize_job_description(text):
    text_lower = text.lower()

    title = extract_field(text, "Job title")
    category = extract_field(text, "Category")
    company = extract_field(text, "Company")
    location = extract_field(text, "Location")
    schedule = extract_field(text, "Schedule")
    skills_field = extract_field(text, "Skills")

    role = category or title or "Data professional"

    summary = f"This job is for a {role}"

    if company:
        summary += f" at {company}"

    if location:
        summary += f" based in {location}"

    summary += "."

    if schedule:
        summary += f" The position is {schedule.lower()}."

    skills_list = [
        "sql", "python", "power bi", "tableau", "excel",
        "pandas", "numpy", "etl", "data warehouse",
        "data warehousing", "data cleaning", "reporting",
        "dashboard", "dashboards", "data visualization",
        "statistics", "statistical analysis", "business intelligence",
        "machine learning", "spark", "aws", "azure", "gcp",
        "clinical data", "data engineering", "airflow"
    ]

    found_skills = []

    if skills_field:
        found_skills.extend(
            [s.strip() for s in re.split(r",|\||;|\[|\]", skills_field) if s.strip()]
        )

    for skill in skills_list:
        if skill in text_lower and skill.title() not in found_skills:
            found_skills.append(skill.upper() if skill in ["sql", "etl", "aws", "gcp"] else skill.title())

    found_skills = list(dict.fromkeys(found_skills))

    if found_skills:
        summary += f" Required skills include: {', '.join(found_skills[:12])}."

    summary += " The candidate will work on data analysis, reporting, dashboards, data quality, and business insights."

    return summary

## 13) Test with your own job description

In [ ]:
my_job_description = """
We are seeking a highly motivated and detail-oriented Senior Data Analyst to join our growing analytics team. The ideal candidate will have strong expertise in SQL, Python, Power BI, and data visualization techniques, with the ability to transform complex datasets into meaningful business insights.

In this role, you will be responsible for collecting, cleaning, analyzing, and interpreting large volumes of structured and unstructured data from multiple business sources. You will design and maintain interactive dashboards and reports using Power BI, helping stakeholders monitor KPIs, operational performance, customer behavior, and financial metrics.

The candidate will work closely with cross-functional teams including marketing, finance, operations, and product management to identify trends, optimize decision-making processes, and support strategic initiatives. You will also participate in data quality improvement projects, automate recurring reporting tasks using Python scripts, and contribute to the development of scalable analytics solutions.

Required qualifications include advanced SQL querying skills, strong experience with Python libraries such as Pandas and NumPy, and hands-on experience building dashboards in Power BI or similar BI tools. Knowledge of ETL processes, data warehousing concepts, and statistical analysis is highly appreciated.

The ideal profile is analytical, proactive, business-oriented, and capable of communicating technical findings clearly to non-technical audiences. Previous experience in business intelligence, reporting automation, or enterprise analytics environments is considered a strong advantage.
"""

print(summarize_job_description(my_job_description))

## 14) Optional: summarize a row from the dataset

In [ ]:
row_number = 0

text = df["job_text"].iloc[row_number]

print("TITLE/TEXT PREVIEW:")
print(text[:800])

print("\nSUMMARY:")
print(summarize_job_description(text))

## Notes

This project now works end-to-end. However, the summaries are trained from pseudo-labels, not human-written labels. For a stronger final project, manually create 100–300 high-quality summaries and fine-tune on them.